# 🫀 실험 20 — **외부검증: PTBDB**. 다른 병원, 그리고 **급성** 심근경색

**MedKOS / `notebooks/exp20_ptbdb_external.ipynb`** · 퀘스트 `ailab-2026-0015`
**PTB Diagnostic ECG Database (PhysioNet Open Access — 자격심사 불필요)**
학습 **6회** · 나머지는 다운로드·전처리

---

## 왜 여기까지 왔나 — 세 번 연속 같은 결론

| 실험 | 결론 |
|---|---|
| 16~18 | **유도**로는 천장. 5전극이 12유도의 0.008 AUROC 이내 |
| 19 | **동작점**으로도 천장. 2단계는 무효(그리고 22 에서 **해롭다**) |
| 22 | 위양성은 **'심전도가 정상이 아니다' 자체**를 따라간다. PTB-XL 은 NORM 이 **43.6%** 라 우리 경보율은 **낙관적** |

세 번 다 같은 곳을 가리킨다 — **남은 변수는 코호트다.**

## PTBDB 가 MIMIC-IV-ECG 를 대신한다

| | PTBDB |
|---|---|
| 접근 | **Open Access** — CITI·추천인 불필요 |
| 규모 | 549 레코드 / 290명 (**MI 148명**) |
| 신호 | 12유도 + Frank XYZ, **1000 Hz** → 100 Hz 로 리샘플 |
| **부위 라벨** | `.hea` 에 `Acute infarction (localization): infero-lateral` |
| **급성/진구성** | `Acute` 와 `Former` 를 **따로** 기록 |
| MI 유병률 | **51%** (vs PTB-XL 의 부위별 0.2~12%) |

**②가 핵심이다.** PTB-XL 의 MI 라벨은 대부분 **진구성 Q파**인데, 우리는 그걸로 배워놓고
"급성 심근경색을 잡는다" 는 말을 한 적이 없다(그래서 카드마다 단서를 달았다).
PTBDB 는 **급성만 골라 평가**할 수 있다 — MIMIC 이 필요했던 진짜 이유가 이것이다.

## 설계에서 조심할 것

**① 라벨 매핑을 추측하지 않는다.** `.hea` 의 국소화 문자열을 **전수 열거**하고,
매핑되지 않는 문자열이 하나라도 있으면 **즉시 멈춘다.** 실험13·13b·15 가
subclass/code 를 혼동해 3개 노트북을 지나갔던 그 실수를 반복하지 않는다.

**② 헤더만 먼저 받는다.** 신호 전체는 ~1.7 GB 다. **`.hea`(약 1 MB)만 먼저 받아
인구조사와 라벨 매핑을 끝내고**, 통과한 뒤에 신호를 받는다. 실패는 1분 안에 난다.

**③ 진폭 스케일을 대조한다.** PTB-XL 캐시와 PTBDB 의 mV 분포가 다르면 성능이 조용히
무너진다. 리샘플 후 **유도 순서(아인트호벤 항등식)와 진폭 분포를 둘 다** 확인한다.

**④ 환자 단위로 집계한다.** 1인당 레코드가 여러 개다. **환자 단위 AUROC 가 주 지표**,
레코드 단위는 부지표.

**⑤ 내부/외부 비교의 비대칭을 명시한다.** 내부는 5겹 OOF, 외부는 **PTB-XL 전량 학습**
모델이다. 외부 쪽이 데이터를 더 봤으므로 **낙폭은 과소추정**된다 — 유리한 쪽으로 기운 비교다.

## 무엇을 얼마나 돌리나

| | 학습 |
|---|---|
| `{I,II,V2,V5}` × 3시드, **PTB-XL 전량** | 3 |
| `{12}` × 3시드, **PTB-XL 전량** | 3 |
| **합** | **6회 (~10분)** + 다운로드·전처리 |

## 사전등록 (결과 보기 전에 고정)

| | 예측 | 성격 |
|---|---|---|
| **G0** | 국소화 문자열 **전수 매핑**(미매핑 0건) · 리샘플 후 유도 순서 항등식 · 진폭 분포 대조 | 전제. 하나라도 실패하면 중단 |
| **P-1 ★★** | 채점 가능 부위 전부에서 **외부 AUROC 낙폭 < 0.10** | 일반화의 기본선 |
| **P-2 ★** | 외부에서도 5전극 `D` = AUROC(12) − AUROC(5전극) **< 0.05** | 배포 사양이 외부에서도 서나 |
| **P-3 ★★ (이 실험의 이유)** | **급성** MI 하위군 AUROC ≥ **진구성** 하위군 − 0.10 | PTB-XL 은 진구성으로 배웠다. **급성에서도 되나** |
| **P-4 ★** | 관측 PPV 가 **베이즈 예측 PPV 의 ±0.15 이내** (내부 민감도·특이도 + 외부 유병률) | **PPV 상승이 유병률만으로 설명되나, 아니면 특이도가 무너지나** |

> **P-4 가 실험22 의 후속이다.** 실험22 는 "위양성이 비정상 자체를 따라간다" 고 했다.
> PTBDB 는 **정상이 52/290 뿐**인 코호트다. 유병률만 오르면 PPV 는 베이즈대로 오르고,
> **특이도가 무너지면 예측보다 낮게 나온다.** 그 차이가 실험22 결론의 실측 검정이다.

> ⚠️ **작다.** MI 148명을 7부위로 나누면 부위당 수십 명이다. **양성 환자 20명 미만
> 부위는 채점에서 빼고 표에만** 남긴다(IPLMI 는 거의 확실히 빠진다).
> `IPLMI`(n=51)에서 반복한 실수를 여기서도 안 한다.

## 이 실험이 무엇을 정하나

- `P-1`·`P-3` ✅ → **논문의 핵심 숫자 두 개**를 확보한다: 외부 낙폭, 급성에서의 성능.
- `P-3` ❌ → **"진구성으로 배운 모델은 급성에 못 쓴다"** 가 정량화된다. 그것도 결과다.
- `P-4` ❌ → 실험22 의 "특이도가 비정상 비율에 무너진다" 가 외부에서 확증된다.


In [ ]:
# CELL 0 — 공용 사전점검
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 적재: assert_label_vocab · decide · assert_arm_shape · "
      "assert_lead_order · boot_indices")

In [ ]:
# CELL 1 — 설정
!pip -q install wfdb

import os, sys, json, time, ast, re, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15~22 와 한 글자도 달라선 안 되는 블록
K_FOLD, EPOCHS, SEED0, NMIN = 5, 20, 20260801, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
LEADS = {"I+II+V2+V5": [0, 1, 7, 10], "12": list(range(12))}
# ★★ 여기까지

DEPLOY, REF = "I+II+V2+V5", "12"
SEEDS   = [0, 1, 2]
FS_IN, FS_OUT, SEC, SKIP = 1000, 100, 10, 1     # 1000Hz → 100Hz · 10초 · 앞 1초 스킵
GMIN_PT = 20            # ★ 양성 **환자** 20명 미만 부위는 채점 제외(표에만)
DROP_THR, D_THR, ACUTE_THR, PPV_TOL = 0.10, 0.05, 0.10, 0.15
BOOT = 2000
PTBDB_URL = "https://physionet.org/files/ptbdb/1.0.0"

# ── PTBDB 국소화 문자열 → 우리 7부위.
#    ★ 여기 없는 문자열이 하나라도 나오면 CELL 2 가 멈춘다(추측 금지).
#    ★ 값은 **리스트**다 — 한 레코드에 두 부위가 함께 적히는 경우가 있다.
def norm_loc(s):
    """소문자·비알파 제거로 표기 흔들림을 흡수한다."""
    return re.sub(r"[^a-z]", "", str(s).strip().lower())

def split_locs(s):
    """★ 통째로 정규화한다 — 구분자로 쪼개지 않는다.

    쪼개면 'n/a' 가 'n'+'a' 로 갈라져 게이트가 또 터진다(테스트에서 잡혔다).
    복합 기재('anterior-inferior' 등)는 **정규화된 통짜 문자열을 LOC_MAP 에 명시**해
    다중 부위로 매핑한다. 새 형태가 나오면 게이트가 잡아주므로 그때 추가하면 된다.
    """
    return [norm_loc(s)]

LOC_MAP = {                       # 정규화 문자열 → 부위 코드 **리스트**
    "anterior": ["AMI"],
    "anteroseptal": ["ASMI"], "anteriorseptal": ["ASMI"],
    "anterolateral": ["ALMI"], "anteriorlateral": ["ALMI"],
    "anteroapicallateral": ["ALMI"],
    "anteroseptallateral": ["ASMI"], "anteroseptolateral": ["ASMI"],
    "inferior": ["IMI"],
    "inferolateral": ["ILMI"], "inferiorlateral": ["ILMI"],
    "inferoposterolateral": ["IPLMI"], "inferoposterlateral": ["IPLMI"],
    "inferiorposteriorlateral": ["IPLMI"],
    "lateral": ["LMI"],
    # ── 2026-08-02 실제 헤더에서 나온 것들. 게이트가 잡아 여기 명시했다.
    #    (아래 표에 **원문 예시**가 함께 찍히므로 눈으로 검증할 것)
    "inferolatera": ["ILMI"],                 # 원본 표기 누락('infero-latera'). 의미는 명백
    "anteriorinferior": ["AMI", "IMI"],       # 전벽 + 하벽 동시 → 다중라벨이므로 둘 다 켠다
    "anterioranterior": ["AMI"],              # 중복 기재
    "inferoposteriorinferior": ["IMI"],       # 하후벽 + 하벽. 후벽(IPMI)은 PTB-XL n<50 로
                                              # 제외된 부위라 IMI 만 남는다
}
LOC_DROP = {                      # 있는 걸 알지만 **우리 7부위에 없는** 것 → 명시적으로 버린다
    "no", "nein", "unknown", "", "none", "na",    # 'na' = n/a
    "inferoposterior", "inferiorposterior",       # IPMI — PTB-XL 에서 n<50 이라 제외됐다
    "posterior", "posterolateral", "posteriorlateral",  # PMI/PLMI — 마찬가지
}

CONFIG = dict(exp="exp20_ptbdb_external", quest="ailab-2026-0015",
              parent_exp=["exp18_confirm", "exp19_two_stage", "exp22_mimic_fp"],
              purpose=("외부 병원·급성 MI 에서의 낙폭을 잰다. MIMIC-IV-ECG 는 credentialed 라 "
                       "Open Access 인 PTBDB 로 대체한다"),
              dataset="PTB Diagnostic ECG Database (PhysioNet, Open Access)",
              change_one_thing="백본·전처리 규격은 실험15~19 와 동일. 평가 데이터만 외부",
              configs=list(LEADS), deploy=DEPLOY, seeds=SEEDS,
              resample=f"{FS_IN}Hz → {FS_OUT}Hz · {SEC}s · 앞 {SKIP}s 스킵",
              unit="환자 단위 집계가 주 지표(1인 다레코드) · 레코드 단위는 부지표",
              asymmetry=("내부는 5겹 OOF, 외부는 PTB-XL 전량 학습 모델이다 — "
                         "외부가 데이터를 더 봤으므로 **낙폭은 과소추정**된다"),
              loc_map=LOC_MAP, loc_drop=sorted(LOC_DROP), gmin_patients=GMIN_PT,
              predictions={
                  "G0": "국소화 문자열 전수 매핑(미매핑 0) · 유도 순서 항등식 · 진폭 분포 대조",
                  "P-1": f"채점 가능 부위 전부에서 외부 AUROC 낙폭 < {DROP_THR}",
                  "P-2": f"외부에서도 5전극 D < {D_THR}",
                  "P-3": f"급성 하위군 AUROC >= 진구성 하위군 - {ACUTE_THR}",
                  "P-4": f"관측 PPV 가 베이즈 예측 PPV 의 ±{PPV_TOL} 이내"},
              caveat=f"양성 환자 {GMIN_PT}명 미만 부위는 채점 제외 — IPLMI 실수 반복 금지",
              k_fold=K_FOLD, epochs=EPOCHS, seed0=SEED0, boot=BOOT)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp20_ptbdb", CONFIG, project=PROJECT)

REG = os.path.join(PROJECT, "registry.jsonl")
DIRS = {}
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") in ("exp18_confirm", "exp19_two_stage") and os.path.isdir(r.get("dir", "")):
        DIRS[r["exp_id"]] = r["dir"]
PARENTS = [DIRS[k] for k in ("exp19_two_stage", "exp18_confirm") if k in DIRS]
run.log("부모 실행: " + (", ".join(f"{k}={v}" for k, v in DIRS.items()) or "없음"))

def arm_at(d, n):
    p = os.path.join(d, "arms", n, "probs.npy")
    return np.load(p) if os.path.exists(p) else None
def find_arm(n):
    for d in PARENTS:
        a = arm_at(d, n)
        if a is not None:
            return a
    return None

In [ ]:
# CELL 2 — 【G0-a】 헤더만 받아 인구조사 + 라벨 매핑 (신호는 아직 안 받는다)
#   ★ 신호 전체는 ~1.7GB 다. 라벨 매핑이 틀리면 그걸 다 받고 나서 알게 된다.
#     .hea 만 먼저 받아 **1분 안에** 실패하게 만든다.
import subprocess, wfdb, pandas as pd

PDB = "/content/ptbdb"; os.makedirs(PDB, exist_ok=True)
recs_f = os.path.join(PDB, "RECORDS")
if not (os.path.exists(recs_f) and os.path.getsize(recs_f) > 0):
    subprocess.run(["wget", "-q", "-O", recs_f, f"{PTBDB_URL}/RECORDS"], check=True)
RECS = [x.strip() for x in open(recs_f) if x.strip()]
run.log(f"PTBDB 레코드 {len(RECS)}건")

t0 = time.time()
need = [r for r in RECS if not os.path.exists(os.path.join(PDB, r + ".hea"))]
if need:
    run.log(f"헤더 {len(need)}개 내려받는 중 (약 1MB)…")
    for i, r in enumerate(need):
        d = os.path.join(PDB, os.path.dirname(r)); os.makedirs(d, exist_ok=True)
        subprocess.run(["wget", "-q", "-c", "-O", os.path.join(PDB, r + ".hea"),
                        f"{PTBDB_URL}/{r}.hea"], check=False)
        if (i + 1) % 150 == 0:
            run.log(f"  {i+1}/{len(need)} · {time.time()-t0:.0f}s")
run.log(f"헤더 준비 완료 {time.time()-t0:.0f}s")

def parse_hea(rec):
    h = wfdb.rdheader(os.path.join(PDB, rec))
    info = {"rec": rec, "patient": rec.split("/")[0],
            "fs": h.fs, "sig_name": [s.lower() for s in h.sig_name],
            "n_sig": h.n_sig, "sig_len": h.sig_len}
    for c in (h.comments or []):
        if ":" in c:
            k, v = c.split(":", 1)
            info[k.strip().lower()] = v.strip()
    return info

H = pd.DataFrame([parse_hea(r) for r in RECS])
run.log(f"헤더 파싱 {len(H)}건 · 환자 {H.patient.nunique()}명")

# ── 【G0-a1】 국소화 문자열 **전수 열거** → 매핑되지 않으면 즉시 중단
ACOL = next((c for c in H.columns if c.startswith("acute infarction")), None)
FCOL = next((c for c in H.columns if c.startswith("former infarction")), None)
RCOL = next((c for c in H.columns if c.startswith("reason for admission")), None)
if ACOL is None or FCOL is None:
    raise RuntimeError(f"헤더에 급성/진구성 국소화 필드가 없습니다. 컬럼: {list(H.columns)}")
run.log(f"필드: 급성='{ACOL}' · 진구성='{FCOL}' · 입원사유='{RCOL}'")

seen, raw_ex = {}, {}
for col in (ACOL, FCOL):
    for v in H[col].fillna(""):
        for k in split_locs(v):
            seen[k] = seen.get(k, 0) + 1
            raw_ex.setdefault(k, set()).add(str(v).strip()[:34])
run.log("\n【국소화 문자열 전수】 정규화 · 건수 · 매핑 · **원문 예시**")
run.log("  ★ 원문 예시를 반드시 눈으로 확인할 것 — 매핑 판단이 맞는지는 여기서만 보인다")
unmapped = []
for k in sorted(seen, key=lambda x: -seen[x]):
    tgt = LOC_MAP.get(k)
    tag = ("→ " + "+".join(tgt)) if tgt else ("(버림)" if k in LOC_DROP else "❌ 미매핑")
    if tag == "❌ 미매핑":
        unmapped.append(k)
    ex = " | ".join(sorted(raw_ex[k])[:3])
    run.log(f"  {k or '(빈칸)':<26}{seen[k]:>5}건  {tag:<16} 원문: {ex}")
if unmapped:
    raise LabelVocabError(
        f"매핑되지 않은 국소화 문자열 {unmapped}.\n"
        + "\n".join(f"    {u!r} 원문 예시: {sorted(raw_ex[u])[:5]}" for u in unmapped)
        + "\n  → **추측해서 넘어가지 않는다.** 위 원문을 보고 LOC_MAP 또는 LOC_DROP 에\n"
          "     명시적으로 넣고 다시 돌린다(헤더만 받은 상태라 1분이면 재실행된다).\n"
        "  (실험13·13b·15 가 subclass/code 혼동으로 3개 노트북을 지나간 실수의 재발 방지)")
run.log("✅ 미매핑 0건")

# ── 부위 라벨 + 급성/진구성 플래그
def sites_of(v):
    """복합 기재를 쪼개 **모든** 부위를 켠다(한 레코드에 두 부위가 적히기도 한다)."""
    out = []
    for k in split_locs(v):
        out += LOC_MAP.get(k, [])
    return sorted(set(out))
H["site_acute"] = H[ACOL].fillna("").apply(sites_of)
H["site_former"] = H[FCOL].fillna("").apply(sites_of)
H["is_acute"] = H.site_acute.apply(bool)
H["is_former"] = H.site_former.apply(bool)
H["sites"] = [sorted(set(a) | set(f)) for a, f in zip(H.site_acute, H.site_former)]
H["is_mi"] = H.sites.apply(bool)

SITES18 = json.load(open(os.path.join(DIRS["exp18_confirm"], "result.json"),
                         encoding="utf-8"))["sites"]
run.log(f"\n실험18 부위 순서(그대로 써야 arm 열이 맞는다): {SITES18}")

run.log("\n【인구조사】 레코드 / 환자")
run.log(f"  {'부위':<8}{'레코드':>8}{'환자':>7}{'급성 환자':>10}{'진구성 환자':>12}{'채점':>7}")
CENSUS = {}
for s in SITES18:
    rmask = H.sites.apply(lambda x: s in x)
    pts = H.loc[rmask, "patient"].nunique()
    pa = H.loc[H.site_acute.apply(lambda x: s in x), "patient"].nunique()
    pf = H.loc[H.site_former.apply(lambda x: s in x), "patient"].nunique()
    CENSUS[s] = {"rec": int(rmask.sum()), "pt": int(pts), "pt_acute": int(pa),
                 "pt_former": int(pf), "scoreable": bool(pts >= GMIN_PT)}
    run.log(f"  {s:<8}{int(rmask.sum()):>8}{pts:>7}{pa:>10}{pf:>12}"
            + ("     ✅" if pts >= GMIN_PT else "   ⚠️제외"))
SCORE_SITES = [s for s in SITES18 if CENSUS[s]["scoreable"]]
n_mi_pt = H.loc[H.is_mi, "patient"].nunique()
run.log(f"\n  MI 환자 {n_mi_pt}명 / 전체 {H.patient.nunique()}명 "
        f"= 유병률 {n_mi_pt / H.patient.nunique():.1%}")
run.log(f"  채점 대상 부위 {SCORE_SITES}")
if not SCORE_SITES:
    raise RuntimeError(f"환자 {GMIN_PT}명 이상인 부위가 하나도 없습니다 — 채점 불가")

In [ ]:
# CELL 3 — 신호 다운로드 + 전처리 + 【G0-b】 유도 순서·진폭 대조
from scipy.signal import resample_poly

SIG_CACHE = run.data("ptbdb_12lead_100hz.npz")
if not os.path.exists(SIG_CACHE):
    run.log("신호 다운로드 (~1.7GB · wget -c 라 끊겨도 이어받는다)")
    t0 = time.time()
    for i, r in enumerate(RECS):
        for ext in (".dat",):
            p = os.path.join(PDB, r + ext)
            if os.path.exists(p) and os.path.getsize(p) > 0:
                continue
            os.makedirs(os.path.dirname(p), exist_ok=True)
            subprocess.run(["wget", "-q", "-c", "-O", p, f"{PTBDB_URL}/{r}{ext}"], check=False)
        if (i + 1) % 50 == 0:
            run.log(f"  {i+1}/{len(RECS)} · {time.time()-t0:.0f}s")
    run.log(f"다운로드 {time.time()-t0:.0f}s · 전처리 시작")

    ORDER = ["i", "ii", "iii", "avr", "avl", "avf",
             "v1", "v2", "v3", "v4", "v5", "v6"]
    n_out = SEC * FS_OUT
    X, keep, bad = [], [], []
    for r in RECS:
        try:
            sig, fields = wfdb.rdsamp(os.path.join(PDB, r))
            names = [s.lower() for s in fields["sig_name"]]
            idx = [names.index(nm) for nm in ORDER]        # 이름으로 찾는다(순서 가정 금지)
            fs = fields["fs"]
            a = int(SKIP * fs); b = a + int(SEC * fs)
            if sig.shape[0] < b:
                bad.append((r, "짧음")); continue
            seg = sig[a:b, idx].astype("float64")
            if not np.isfinite(seg).all():
                bad.append((r, "NaN")); continue
            g = int(round(fs / FS_OUT))
            seg = resample_poly(seg, 1, g, axis=0) if g > 1 else seg
            if seg.shape[0] != n_out:
                bad.append((r, f"길이 {seg.shape[0]}")); continue
            X.append(seg.astype("float32")); keep.append(r)
        except Exception as e:
            bad.append((r, str(e)[:40]))
    X = np.stack(X)
    run.log(f"전처리 완료 X{X.shape} · 실패 {len(bad)}건" + (f" 예: {bad[:3]}" if bad else ""))
    np.savez_compressed(SIG_CACHE, X=X, recs=np.array(keep))
    del X
z = np.load(SIG_CACHE, allow_pickle=True)
XE, RKEEP = z["X"], [str(x) for x in z["recs"]]
run.log(f"PTBDB 캐시 {XE.shape} · 레코드 {len(RKEEP)}")

# 【G0-b1】 유도 순서 — 리샘플 후에도 아인트호벤 항등식이 서는가
chk = assert_lead_order(XE)
run.log("【G0-b1】 유도 순서 ✅ " + " · ".join(f"{k} {v:.4f}" for k, v in chk["rel_err"].items()))

# 【G0-b2】 진폭 스케일 — PTB-XL 캐시와 분포가 다르면 성능이 조용히 무너진다
PX = run.data("ptbxl_12lead_all.npz")
if not os.path.exists(PX):
    raise RuntimeError(f"PTB-XL 전량 캐시가 없습니다: {PX}")
zx = np.load(PX, allow_pickle=True)
XI, FOLD10, EID = zx["X"], zx["fold"], zx["eid"]
CV = (FOLD10 - 1) % K_FOLD
rs_ = np.random.RandomState(SEED0)
si = rs_.choice(len(XI), 500, replace=False); se = rs_.choice(len(XE), min(500, len(XE)), replace=False)
run.log("\n【G0-b2】 진폭 분포 (표본 500) — 유도별 표준편차(mV)")
run.log(f"  {'유도':<6}{'PTB-XL':>10}{'PTBDB':>10}{'비':>8}")
ratios = []
for j, nm in enumerate(["I", "II", "III", "aVR", "aVL", "aVF",
                        "V1", "V2", "V3", "V4", "V5", "V6"]):
    a = float(XI[si, :, j].std()); b = float(XE[se, :, j].std())
    ratios.append(b / a if a else np.nan)
    run.log(f"  {nm:<6}{a:>10.4f}{b:>10.4f}{b/a if a else np.nan:>8.2f}")
SCALE = float(np.nanmedian(ratios))
run.log(f"  → 배수 중앙값 {SCALE:.2f}")
if not (0.5 <= SCALE <= 2.0):
    run.log("  ⛔ 진폭 스케일이 2배 넘게 다르다 — 이대로 넣으면 성능이 무의미해진다. 중단한다")
    raise RuntimeError(f"진폭 스케일 불일치 {SCALE:.2f}")
run.log("  ✅ 같은 단위계로 본다(보정 없이 그대로 넣는다 — 임의 보정은 새 자유도다)")

# 부위 라벨(레코드 단위) — SITES18 순서 그대로
HK = H.set_index("rec").loc[RKEEP]
YE = np.stack([[s in row for s in SITES18] for row in HK.sites]).astype(bool)
PT = HK.patient.values
AC = HK.is_acute.values; FM = HK.is_former.values
YEA = np.stack([[s in row for s in SITES18] for row in HK.site_acute]).astype(bool)
YEF = np.stack([[s in row for s in SITES18] for row in HK.site_former]).astype(bool)
run.log(f"\n외부 라벨 준비 · 레코드 {YE.shape} · 환자 {len(set(PT))}명")

In [ ]:
# CELL 4 — PTB-XL **전량** 학습 (6회) → 외부 적용
import tensorflow as tf
from tensorflow.keras import layers, models

dfx = pd.read_csv(os.path.join("/content/ptbxl", "ptbxl_database.csv"), index_col="ecg_id") \
    if os.path.exists("/content/ptbxl/ptbxl_database.csv") else None
if dfx is None:
    os.makedirs("/content/ptbxl", exist_ok=True)
    subprocess.run(["wget", "-q", "-O", "/content/ptbxl/ptbxl_database.csv",
                    "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"], check=True)
    dfx = pd.read_csv("/content/ptbxl/ptbxl_database.csv", index_col="ecg_id")
dfa = dfx.loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
vocab = {c for cs in dfa.codes for c in cs}
assert_label_vocab(SITES18, vocab, kind="MI 부위 코드")
YI = np.stack([[s in c for s in SITES18] for c in dfa.codes]).astype("float32")
NS = len(SITES18)
run.log(f"내부 학습 라벨 {YI.shape} · 부위 {SITES18}")

MASKS = {c: np.zeros(12, "float32") for c in LEADS}
for c, idx in LEADS.items():
    MASKS[c][idx] = 1.0

def build_head(seed):
    """★ 실험15~19 와 완전히 동일."""
    tf.keras.utils.set_random_seed(seed)
    si_ = layers.Input((XI.shape[1], 12))
    x = si_
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(si_, layers.Dense(NS, activation="sigmoid")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="binary_crossentropy")
    return m

# 전량 학습이므로 겹이 없다. 검증셋만 내부에서 떼어 조기 종료 없이 에폭 고정(실험15~19 와 동일).
rs2 = np.random.RandomState(SEED0); order = rs2.permutation(len(XI))
n_val = max(int(len(order) * 0.12), 200)
VA, TR = order[:n_val], order[n_val:]
run.log(f"전량 학습 — 학습 {len(TR):,} · 검증 {len(VA):,}")

PE = {c: {} for c in LEADS}        # 외부 예측
t0, done = time.time(), 0
for c in LEADS:
    mk = MASKS[c]
    for sd in SEEDS:
        a = run.load_arm(f"ext_{c}_s{sd}")
        if a is None:
            m = build_head(SEED0 + 15 + sd)          # ★ 겹이 없으므로 100*k 항이 빠진다
            m.fit(XI[TR] * mk, YI[TR], validation_data=(XI[VA] * mk, YI[VA]),
                  epochs=EPOCHS, batch_size=128, verbose=0)
            a = m.predict(XE * mk, batch_size=512, verbose=0)
            run.save_arm(f"ext_{c}_s{sd}", a)
            if sd == SEEDS[0]:
                run.save_model(m, f"full_{c}")
            tf.keras.backend.clear_session(); done += 1
            run.log(f"  {c:<12} 시드{sd} 학습 완료 ({done} · {time.time()-t0:.0f}s)")
        assert_arm_shape(a, len(XE), name=f"ext_{c}_s{sd}")
        PE[c][sd] = a
run.log(f"총 {time.time()-t0:.0f}s · 이번 세션 학습 {done}회")

In [ ]:
# CELL 5 — 외부 평가: 환자 단위 AUROC · 낙폭 · 급성 vs 진구성
from sklearn.metrics import roc_auc_score
from scipy import stats

def t_ci(v, conf=.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2: return m, np.nan, np.nan, 0.0
    sd = float(v.std(ddof=1))
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * sd / np.sqrt(n))
    return m, m - h, m + h, sd

PTS = sorted(set(PT))
PIDX = {p: np.where(PT == p)[0] for p in PTS}

def by_patient(score, y):
    """★ 환자 단위 집계 — 1인 다레코드. 예측은 평균, 라벨은 any."""
    s = np.array([score[PIDX[p]].mean() for p in PTS])
    t = np.array([y[PIDX[p]].any() for p in PTS])
    return s, t

def ext_auc(c, sd, j, ymat=None, keep=None):
    y = (ymat if ymat is not None else YE)[:, j]
    sc = PE[c][sd][:, j]
    if keep is not None:
        sc, y = sc[keep], y[keep]
        s, t = sc, y
    else:
        s, t = by_patient(sc, y)
    return float(roc_auc_score(t, s)) if t.any() and (~t).any() else np.nan

# ── 내부 기준: 실험18·19 의 저장 arm (5겹 OOF, 시드 0~2)
def int_auc(c, sd, j):
    o = np.zeros(len(EID), "float32")
    for k in range(K_FOLD):
        a = find_arm(f"{c}_s{sd}_f{k}")
        if a is None:
            return np.nan
        o[np.where(CV == k)[0]] = a[:, j]
    return float(roc_auc_score(YI[:, j].astype(bool), o))

run.log("\n" + "=" * 118)
run.log("【외부검증】 환자 단위 AUROC · 낙폭 = 내부 − 외부")
run.log("=" * 118)
run.log(f"  {'부위':<8}{'환자':>6}{'급성':>6}{'진구성':>8}"
        + "".join(f"{'내부 ' + c:>16}{'외부 ' + c:>16}{'낙폭':>9}" for c in [DEPLOY]))
INT, EXT, DROP = {}, {}, {}
for j, s in enumerate(SITES18):
    for c in LEADS:
        INT[(c, s)] = [int_auc(c, sd, j) for sd in SEEDS]
        EXT[(c, s)] = [ext_auc(c, sd, j) for sd in SEEDS]
    DROP[s] = [INT[(DEPLOY, s)][i] - EXT[(DEPLOY, s)][i] for i in range(len(SEEDS))]
    run.log(f"  {s:<8}{CENSUS[s]['pt']:>6}{CENSUS[s]['pt_acute']:>6}"
            f"{CENSUS[s]['pt_former']:>8}"
            f"{np.nanmean(INT[(DEPLOY, s)]):>16.4f}{np.nanmean(EXT[(DEPLOY, s)]):>16.4f}"
            f"{np.nanmean(DROP[s]):>+9.4f}"
            + ("" if s in SCORE_SITES else "   ⚠️채점제외"))

run.log(f"\n【5전극 잔여 결손 D】 내부 vs 외부")
run.log(f"  {'부위':<8}{'내부 D':>10}{'외부 D':>10}")
DI, DE = {}, {}
for s in SITES18:
    DI[s] = [INT[(REF, s)][i] - INT[(DEPLOY, s)][i] for i in range(len(SEEDS))]
    DE[s] = [EXT[(REF, s)][i] - EXT[(DEPLOY, s)][i] for i in range(len(SEEDS))]
    run.log(f"  {s:<8}{np.nanmean(DI[s]):>+10.4f}{np.nanmean(DE[s]):>+10.4f}")

# ── 급성 vs 진구성 (환자 단위)
run.log("\n【급성 vs 진구성】 그 부위 양성을 급성/진구성으로 나눠 각각 AUROC")
run.log("  ★ 음성군은 **MI 가 전혀 없는 환자**로 공통 고정 — 두 하위군을 같은 잣대로 잰다")
no_mi_pt = np.array([not HK.is_mi.values[PIDX[p]].any() for p in PTS])
ACU, FOR = {}, {}
for j, s in enumerate(SITES18):
    for tag, ymat, box in (("acute", YEA, ACU), ("former", YEF, FOR)):
        pos = np.array([ymat[PIDX[p], j].any() for p in PTS])
        keep = pos | no_mi_pt
        vals = []
        for sd in SEEDS:
            sc = np.array([PE[DEPLOY][sd][PIDX[p], j].mean() for p in PTS])
            vals.append(float(roc_auc_score(pos[keep], sc[keep]))
                        if pos[keep].any() and (~pos[keep]).any() else np.nan)
        box[s] = vals
    na = int(sum(YEA[PIDX[p], j].any() for p in PTS))
    nf = int(sum(YEF[PIDX[p], j].any() for p in PTS))
    run.log(f"  {s:<8}급성 n={na:>3} AUROC {np.nanmean(ACU[s]):>7.4f}   "
            f"진구성 n={nf:>3} AUROC {np.nanmean(FOR[s]):>7.4f}   "
            f"차 {np.nanmean(ACU[s]) - np.nanmean(FOR[s]):>+7.4f}")

In [ ]:
# CELL 6 — 【P-4】 PPV: 베이즈 예측 vs 관측
run.log("\n" + "=" * 110)
run.log("【P-4】 PPV — 유병률만으로 설명되나, 아니면 특이도가 무너지나")
run.log("=" * 110)
run.log("  내부에서 민감도 0.90 을 주는 임계값과 그때의 특이도를 가져와,")
run.log("  **외부 유병률**을 넣어 베이즈로 PPV 를 예측하고 **관측 PPV** 와 비교한다.")

def thr_sens(score, pos, t=0.90):
    p = score[pos]
    return float(np.quantile(p, 1.0 - t, method="lower")) if len(p) else -np.inf

PPV = {}
run.log(f"\n  {'부위':<8}{'외부 유병률':>11}{'내부 민감도':>11}{'내부 특이도':>11}"
        f"{'예측 PPV':>10}{'관측 PPV':>10}{'관측 민감도':>12}{'차':>9}")
for j, s in enumerate(SITES18):
    pv, pd_, po, se_, sp_ = [], [], [], [], []
    for sd in SEEDS:
        # 내부: 전량 학습 모델을 내부 검증셋에 적용해 임계값·특이도를 구한다
        #      (OOF arm 은 5겹 모델이라 전량 모델과 임계값이 다르다 — 같은 모델로 맞춘다)
        m_in = PE  # placeholder to keep names tidy
        oof = np.zeros(len(EID), "float32"); ok = True
        for k in range(K_FOLD):
            a = find_arm(f"{DEPLOY}_s{sd}_f{k}")
            if a is None:
                ok = False; break
            oof[np.where(CV == k)[0]] = a[:, j]
        if not ok:
            continue
        yi = YI[:, j].astype(bool)
        th = thr_sens(oof, yi, 0.90)
        sp = float((oof[~yi] < th).mean()); se = float((oof[yi] >= th).mean())
        # 외부(환자 단위)
        sc, yt = by_patient(PE[DEPLOY][sd][:, j], YE[:, j])
        prev = float(yt.mean())
        alarm = sc >= th
        obs = float((alarm & yt).sum() / max(alarm.sum(), 1))
        pred = se * prev / max(se * prev + (1 - sp) * (1 - prev), 1e-9)
        pv.append(prev); pd_.append(pred); po.append(obs)
        se_.append(float((alarm & yt).sum() / max(yt.sum(), 1))); sp_.append(sp)
    if not pd_:
        continue
    PPV[s] = {"prev": float(np.mean(pv)), "pred": float(np.mean(pd_)),
              "obs": float(np.mean(po)), "sens_int": float(np.mean([0.90] * len(pd_))),
              "spec_int": float(np.mean(sp_)), "sens_ext": float(np.mean(se_)),
              "diff": [po[i] - pd_[i] for i in range(len(po))]}
    run.log(f"  {s:<8}{np.mean(pv):>11.3f}{0.90:>11.3f}{np.mean(sp_):>11.3f}"
            f"{np.mean(pd_):>10.1%}{np.mean(po):>10.1%}{np.mean(se_):>12.3f}"
            f"{np.mean(po) - np.mean(pd_):>+9.3f}"
            + ("" if s in SCORE_SITES else "  ⚠️제외"))
run.log("\n  ※ 임계값은 **내부에서** 정하고 외부에 그대로 적용한다(재조정하면 외부검증이 아니다)")

In [ ]:
# CELL 7 — 사전등록 채점
run.log("\n" + "=" * 110)
run.log(f"【사전등록 채점】 시드 t-CI · 채점 대상 {SCORE_SITES}")
run.log("=" * 110)
V = {}

# P-1 낙폭
p1 = []
run.log(f"\n  P-1 외부 AUROC 낙폭 < {DROP_THR}")
for s in SCORE_SITES:
    m, lo, hi, sd = t_ci(DROP[s])
    v = decide(lo, hi, DROP_THR, "<"); p1.append(v)
    run.log(f"      {s:<8}낙폭 {m:>+7.4f} [{lo:+.4f}, {hi:+.4f}] (SD {sd:.4f}) → {MARK[v]}")
V["P-1"] = True if all(x is True for x in p1) else \
    False if any(x is False for x in p1) else None
worst = max(SCORE_SITES, key=lambda s: np.nanmean(DROP[s]))
run.log(f"  P-1 → {MARK[V['P-1']]}  최악 {worst} {np.nanmean(DROP[worst]):+.4f}")

# P-2 외부 D
p2 = []
run.log(f"\n  P-2 외부 5전극 D < {D_THR}")
for s in SCORE_SITES:
    m, lo, hi, sd = t_ci(DE[s])
    v = decide(lo, hi, D_THR, "<"); p2.append(v)
    run.log(f"      {s:<8}D {m:>+7.4f} [{lo:+.4f}, {hi:+.4f}] → {MARK[v]}")
V["P-2"] = True if all(x is True for x in p2) else \
    False if any(x is False for x in p2) else None
run.log(f"  P-2 → {MARK[V['P-2']]}")

# P-3 ★★ 급성
run.log(f"\n  P-3 급성 AUROC >= 진구성 − {ACUTE_THR}  ★ 이 실험의 이유")
p3, use3 = [], []
for s in SCORE_SITES:
    if not (np.isfinite(np.nanmean(ACU[s])) and np.isfinite(np.nanmean(FOR[s]))):
        run.log(f"      {s:<8}급성 또는 진구성 하위군이 비어 판정 불가"); continue
    d = [FOR[s][i] - ACU[s][i] for i in range(len(SEEDS))]   # 양수면 급성이 나쁘다
    m, lo, hi, sd = t_ci(d)
    v = decide(lo, hi, ACUTE_THR, "<"); p3.append(v); use3.append(s)
    run.log(f"      {s:<8}진구성−급성 {m:>+7.4f} [{lo:+.4f}, {hi:+.4f}] → {MARK[v]}")
V["P-3"] = (True if p3 and all(x is True for x in p3) else
            False if any(x is False for x in p3) else None)
run.log(f"  P-3 → {MARK[V['P-3']]}  (판정 부위 {use3})")

# P-4 PPV
run.log(f"\n  P-4 관측 PPV 가 베이즈 예측의 ±{PPV_TOL} 이내")
p4 = []
for s in SCORE_SITES:
    if s not in PPV:
        continue
    m, lo, hi, sd = t_ci(PPV[s]["diff"])
    inside = (abs(lo) <= PPV_TOL and abs(hi) <= PPV_TOL)
    outside = (lo > PPV_TOL or hi < -PPV_TOL)
    v = True if inside else (False if outside else None)
    p4.append(v)
    run.log(f"      {s:<8}관측−예측 {m:>+7.3f} [{lo:+.3f}, {hi:+.3f}] → {MARK[v]}"
            f"   (예측 {PPV[s]['pred']:.1%} · 관측 {PPV[s]['obs']:.1%})")
V["P-4"] = True if p4 and all(x is True for x in p4) else \
    False if any(x is False for x in p4) else None
run.log(f"  P-4 → {MARK[V['P-4']]}")
run.log("      해석: 관측 < 예측 이면 **특이도가 외부에서 무너진 것**"
        "(실험22 의 '위양성은 비정상 자체를 따라간다' 가 외부에서 확증)")

run.log("\n" + "=" * 110)
for k in ("P-1", "P-2", "P-3", "P-4"):
    run.log(f"  {k}: {MARK[V.get(k)]}")
run.log("  ⚠️ 내부는 5겹 OOF · 외부는 전량 학습 모델 — **낙폭은 과소추정**된다(유리한 쪽 비교)")
run.log("=" * 110)

In [ ]:
# CELL 8 — 그림
import matplotlib.pyplot as plt
ss = SCORE_SITES
xs = np.arange(len(ss)); w = .38
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))

ax[0].bar(xs - w/2, [np.nanmean(INT[(DEPLOY, s)]) for s in ss], w, label="내부(PTB-XL)", color="0.6")
ax[0].bar(xs + w/2, [np.nanmean(EXT[(DEPLOY, s)]) for s in ss], w, label="외부(PTBDB)", color="tab:red")
ax[0].axhline(0.5, color="k", lw=1, ls=":")
ax[0].set_xticks(xs); ax[0].set_xticklabels(ss, rotation=45, ha="right")
ax[0].set_ylabel("AUROC (환자 단위)"); ax[0].legend(fontsize=8)
ax[0].set_title(f"외부 낙폭 · P-1 {MARK[V['P-1']]}")

ax[1].bar(xs - w/2, [np.nanmean(FOR[s]) for s in ss], w, label="진구성", color="0.6")
ax[1].bar(xs + w/2, [np.nanmean(ACU[s]) for s in ss], w, label="급성", color="tab:orange")
ax[1].axhline(0.5, color="k", lw=1, ls=":")
ax[1].set_xticks(xs); ax[1].set_xticklabels(ss, rotation=45, ha="right")
ax[1].set_ylabel("AUROC"); ax[1].legend(fontsize=8)
ax[1].set_title(f"급성 vs 진구성 · P-3 {MARK[V['P-3']]}")

pv = [s for s in ss if s in PPV]
if pv:
    x2 = np.arange(len(pv))
    ax[2].bar(x2 - w/2, [PPV[s]["pred"] for s in pv], w, label="베이즈 예측", color="0.6")
    ax[2].bar(x2 + w/2, [PPV[s]["obs"] for s in pv], w, label="관측", color="tab:blue")
    ax[2].set_xticks(x2); ax[2].set_xticklabels(pv, rotation=45, ha="right")
    ax[2].set_ylabel("PPV"); ax[2].legend(fontsize=8)
    ax[2].set_title(f"PPV — 유병률만으로 설명되나 · P-4 {MARK[V['P-4']]}")
else:
    ax[2].axis("off")

plt.tight_layout(); run.save_fig("ptbdb_external", fig); plt.show()

In [ ]:
# CELL 9 — 결과 저장
res = {
    "week": 2, "exp_id": "exp20_ptbdb", "quest": "ailab-2026-0015",
    "task": "외부검증 PTBDB — 다른 병원, 그리고 급성 심근경색",
    "split": "external", "step": "exp20-ptbdb-external",
    "dataset": "PTB Diagnostic ECG Database (Open Access)",
    "metric": f"external_auroc_drop_worst_{DEPLOY}",
    "value": round(float(np.nanmean(DROP[worst])), 4),
    "passed": bool(V.get("P-1") is True and V.get("P-3") is True),
    "date": time.strftime("%Y-%m-%d"),
    "training_runs": int(done), "n_records": int(len(RKEEP)),
    "n_patients": int(len(PTS)), "mi_prevalence_patient": round(
        float(np.mean([HK.is_mi.values[PIDX[p]].any() for p in PTS])), 4),
    "sites": SITES18, "score_sites": SCORE_SITES, "census": CENSUS,
    "amplitude_scale_ratio": round(SCALE, 3),
    "lead_order_rel_err": chk["rel_err"],
    "P-1": V.get("P-1"), "P-2": V.get("P-2"), "P-3": V.get("P-3"), "P-4": V.get("P-4"),
    "worst_drop_site": worst,
    "auroc_internal": {s: round(float(np.nanmean(INT[(DEPLOY, s)])), 4) for s in SITES18},
    "auroc_external": {s: round(float(np.nanmean(EXT[(DEPLOY, s)])), 4) for s in SITES18},
    "auroc_drop": {s: round(float(np.nanmean(DROP[s])), 4) for s in SITES18},
    "D_internal": {s: round(float(np.nanmean(DI[s])), 4) for s in SITES18},
    "D_external": {s: round(float(np.nanmean(DE[s])), 4) for s in SITES18},
    "auroc_acute": {s: round(float(np.nanmean(ACU[s])), 4) for s in SITES18},
    "auroc_former": {s: round(float(np.nanmean(FOR[s])), 4) for s in SITES18},
    "ppv": {s: {k: (round(v, 4) if isinstance(v, float) else v)
                for k, v in PPV[s].items() if k != "diff"} for s in PPV},
    "caveat_asymmetry": ("내부는 5겹 OOF · 외부는 PTB-XL 전량 학습 모델 — "
                         "외부가 데이터를 더 봤으므로 낙폭은 과소추정"),
    "verdict": " · ".join(f"{k} {MARK[V.get(k)]}" for k in ("P-1", "P-2", "P-3", "P-4")),
}
res["summary"] = (f"최악 낙폭 {res['value']:+.4f}({worst}) · "
                  f"MI 유병률 {res['mi_prevalence_patient']:.1%} · " + res["verdict"])
run.save_json("result.json", res)
run.log("\n" + json.dumps({k: res[k] for k in
                           ("metric", "value", "passed", "P-1", "P-2", "P-3", "P-4",
                            "mi_prevalence_patient", "summary")},
                          ensure_ascii=False, indent=2))
run.finish(res)